# Exception Handling

Anticipate runtime errors and handle expected failure conditions without hiding bugs.

> 📘 **Learning path:** Python Foundations · Notebook 17 of 18

## Learning objectives

- Recognize common runtime exceptions.
- Use `try`, `except`, `else`, and `finally`.
- Raise built-in and custom exceptions appropriately.
- Preserve exception context and write helpful error messages.
- Avoid overly broad or silent exception handling.

## How to use this notebook

1. Read each explanation before running its code cell.
2. Run the examples in order and change the values to observe what happens.
3. Complete the practice tasks before opening the suggested solutions at the end.

**Requirements:** Python 3.10+ and a Jupyter-compatible environment. The examples use the Python standard library unless stated otherwise.

## 1. What is an exception?

An exception is a runtime event that interrupts the normal flow of a program. Examples include `ValueError` for an invalid conversion, `ZeroDivisionError` for division by zero, and `FileNotFoundError` for a missing file. Exception handling lets you respond to expected failure conditions.

## 2. Use `try` and `except`

Place statements that may fail in a `try` block and handle the specific exception types you expect in `except` blocks. Avoid catching every exception unless you have a deliberate reason and a meaningful recovery strategy.

In [1]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

5.0
None


## 3. Handle more than one expected error

Different exception types may need different messages or recovery steps. Keep the `try` block as narrow as practical so that the `except` clause does not accidentally hide an unrelated problem.

In [2]:
samples = ["42", "3.5", "not numeric"]
for sample in samples:
    try:
        value = int(sample)
    except ValueError as error:
        print(f"Cannot parse {sample!r} as an integer: {error}")
    else:
        print("Parsed:", value)

Parsed: 42
Cannot parse '3.5' as an integer: invalid literal for int() with base 10: '3.5'
Cannot parse 'not numeric' as an integer: invalid literal for int() with base 10: 'not numeric'


## 4. Understand `else` and `finally`

The `else` block runs only if the `try` block completes without an exception. The `finally` block runs whether an exception occurred or not, making it useful for cleanup. Context managers such as `with open(...)` are often an even safer way to manage resources.

In [3]:
def demonstrate_flow(should_fail):
    try:
        print("Inside try")
        if should_fail:
            raise ValueError("A demonstration error")
    except ValueError as error:
        print("Handled:", error)
    else:
        print("No exception occurred")
    finally:
        print("Cleanup step")

demonstrate_flow(False)
demonstrate_flow(True)

Inside try
No exception occurred
Cleanup step
Inside try
Handled: A demonstration error
Cleanup step


## 5. Raise exceptions when needed

Use `raise` to signal that an operation cannot proceed under the current conditions. Choose an appropriate built-in exception, and include a message that helps the caller understand what went wrong.

In [4]:
def calculate_discount(price, rate):
    if price < 0:
        raise ValueError("price must be non-negative")
    if not 0 <= rate <= 1:
        raise ValueError("rate must be between 0 and 1")
    return price * (1 - rate)

print(calculate_discount(100, 0.15))

85.0


## 6. Define a custom exception

A custom exception can represent a domain-specific failure. Derive it from `Exception` (directly or indirectly) and give it a meaningful name. Avoid creating a custom exception when a standard one already expresses the problem clearly.

In [5]:
class DataValidationError(Exception):
    """Raised when an input record fails domain-specific validation."""

def validate_site_id(site_id):
    if not site_id.startswith("SITE-"):
        raise DataValidationError("site_id must start with 'SITE-'")
    return site_id

for value in ("SITE-104", "104"):
    try:
        print(validate_site_id(value))
    except DataValidationError as error:
        print("Invalid record:", error)

SITE-104
Invalid record: site_id must start with 'SITE-'


## 7. Best practices

- Catch specific exceptions that you can handle meaningfully.
- Do not silently ignore an exception. At minimum, provide useful context or re-raise it.
- Avoid using exceptions for ordinary control flow when a simple condition is clearer.
- Preserve the original cause when translating one exception into another using `raise NewError(...) from error`.
- Do not expose sensitive data in exception messages or logs.

In [6]:
def parse_measurement(text):
    try:
        return float(text)
    except ValueError as error:
        raise ValueError(f"Measurement must be numeric; received {text!r}") from error

print(parse_measurement("12.8"))

12.8


## Common mistakes

- A broad `except:` can hide bugs and should generally be avoided.
- The `else` block is not the same as `finally`: `else` runs only on success; `finally` runs either way.
- Do not catch an exception if you cannot recover, log useful context, or add meaningful value. Letting the error propagate can be the correct choice.

## 7. Catch specific exceptions

Catch the narrowest exception you can handle. `ValueError` often means a value has the right general type but an invalid representation; `TypeError` often indicates an operation was given an inappropriate type. Avoid silently ignoring all exceptions because it can hide real bugs.

In [7]:
def parse_year(text):
    try:
        year = int(text)
    except ValueError:
        print("Year must contain digits in integer form.")
        return None
    if year < 0:
        print("Year cannot be negative.")
        return None
    return year

print(parse_year("2026"))
print(parse_year("twenty"))
print(parse_year("-1"))

2026
Year must contain digits in integer form.
None
Year cannot be negative.
None


## 8. `else` and `finally`

The `else` clause runs only when the `try` block completes without an exception. The `finally` clause runs whether or not an exception occurs, and is commonly used for cleanup. Context managers such as `with open(...)` are usually preferable for resource cleanup.

In [8]:
def demonstrate(value):
    try:
        result = 10 / value
    except ZeroDivisionError:
        print("Cannot divide by zero.")
    else:
        print("Result:", result)
    finally:
        print("Calculation attempt finished.")

demonstrate(2)
demonstrate(0)

Result: 5.0
Calculation attempt finished.
Cannot divide by zero.
Calculation attempt finished.


## 9. Raise clear errors and define custom exceptions

Use `raise` when an input violates a function's contract. A custom exception can make domain-specific failures easier to understand. Keep the error message actionable and avoid catching an exception only to immediately hide it.

In [9]:
class InvalidCapacityError(ValueError):
    """Raised when a capacity is outside the allowed range."""


def validate_capacity(capacity):
    if capacity < 0:
        raise InvalidCapacityError("Capacity must be zero or greater.")
    return capacity

for candidate in [12, -1]:
    try:
        print(validate_capacity(candidate))
    except InvalidCapacityError as error:
        print(type(error).__name__ + ":", error)

12
InvalidCapacityError: Capacity must be zero or greater.


### Preserve the original cause

When translating one exception into a more useful application-level error, use `raise NewError(...) from error` to preserve the underlying cause. This makes debugging more informative.

In [10]:
def load_count(text):
    try:
        return int(text)
    except ValueError as error:
        raise ValueError("Count must be written as an integer.") from error

try:
    load_count("many")
except ValueError as error:
    print(error)
    print("Original cause:", type(error.__cause__).__name__)

Count must be written as an integer.
Original cause: ValueError


## 🧩 Practice tasks

Try to solve these tasks without looking at the solutions. Write your attempts in new code cells below each task.

1. Convert a list of numeric strings to floats, reporting invalid values without stopping the entire loop.
2. Write a function that raises `ValueError` when a number is negative.
3. Define a custom exception for an invalid project code and use it in validation.
4. Explain the difference between `else` and `finally` in a `try` statement.

## ✅ Suggested solutions

These are sample solutions, not the only correct answers. Try the exercises first, then compare your approach.

In [11]:
def safe_divide(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None

print(safe_divide(10, 2))
print(safe_divide(10, 0))

try:
    number = int("not a number")
except ValueError as error:
    print(f"Could not convert input: {error}")

5.0
None
Could not convert input: invalid literal for int() with base 10: 'not a number'


## Key takeaways

- Focus on understanding what the code does, not memorizing every example.
- Make small changes and predict the result before running the cell.
- When an error appears, read its type and message; debugging is part of learning.

**Next notebook:** Object Oriented Programming.